In [ ]:
%load_ext autoreload
%autoreload 2


from src.config import Configuration
import numpy as np
CONFIG = Configuration()

# Load ViT and Data

Set `MODEL_SOURCE = "google"` (HuggingFace) or `MODEL_SOURCE = "timm"` (PyTorch Image Models). Both produce `layer_attentions` of shape `(1, 12, 197, 197)` per layer — the rest of the notebook works identically.

1. **Zebra** (340) - High contrast stripes.
2. **Dalmatian** (251) - Distinct spot patterns.
3. **Monarch butterfly** (323) - Clear shape and contrast against nature backgrounds.
4. **Acoustic guitar** (401) - Strong geometric lines.
5. **School bus** (779) - Large, uniform color block.
6. **Volcano** (980) - Distinct peak structure.
7. **Daisy** (985) - Radial symmetry.
8. **Peacock** (84) - Complex, detailed texture.
9. **Brain coral** (109) - High-frequency texture testing.
10. **Espresso maker** (550) - Metallic, reflective, complex object.

In [ ]:
import os
from maikol_utils.file_utils import list_dir_files

all_images, _ = list_dir_files(CONFIG.attention_data, absolute_path=True)
classes = set([os.path.basename(img).split("_")[0] for img in all_images])

classes

In [ ]:
# MODEL_SOURCE = "google" 
MODEL_SOURCE = "google" 

if MODEL_SOURCE == "google":
    from transformers import ViTModel
    model = ViTModel.from_pretrained(
        'google/vit-base-patch16-224', 
        output_attentions=True # NOTE
    )
elif MODEL_SOURCE == "timm":
    import timm
    model = timm.create_model('vit_base_patch16_224', pretrained=True)

model.eval()

# Visualization

### Get attention

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from types import SimpleNamespace

to_tensor = transforms.ToTensor()

def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((224, 224)) 
    image = to_tensor(image).unsqueeze(0)  # (1, 3, H, W)
    return image


if MODEL_SOURCE == "google":
    with torch.no_grad():
        outputs = {
            path: model(load_image(path)) for path in all_images
        }
    layer_attentions = outputs[all_images[0]].attentions
elif MODEL_SOURCE == "timm":
    _timm_attentions = []
    _timm_num_heads = 12
    _timm_head_dim = 64

    def _timm_attn_hook(module, input, output):
        qkv = output.detach().cpu()
        B, N, C = qkv.shape
        qkv = qkv.reshape(B, N, 3, _timm_num_heads, _timm_head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scale = _timm_head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        _timm_attentions.append(attn)

    _timm_handles = []
    for name, module in model.named_modules():
        if '.qkv' in name and 'attn.qkv' in name:
            _timm_handles.append(module.register_forward_hook(_timm_attn_hook))

    outputs = {}
    with torch.no_grad():
        for path in all_images:
            _timm_attentions.clear()
            model(load_image(path))
            outputs[path] = SimpleNamespace(attentions=[a for a in _timm_attentions])

    for h in _timm_handles:
        h.remove()

    layer_attentions = outputs[all_images[0]].attentions

print("Number of attention layers:", len(layer_attentions)) 
print("Shape of the the attention:", layer_attentions[1].shape) 
layer_attentions

### Layer CLS token attentio over all the paches at layer $l$

- We have 224×224 image separed into 14×14 paches: total of 196 paches.
- We had 197 bc it was $196 \text{(PATCHES)} + 1 \text{(CLS)}$

In [ ]:
from typing import Literal
from maikol_utils.print_utils import print_separator

def full_attention_at_layer(attention, layer_idx, type: Literal["mean", "max", "heads"] = "mean"):
    if type == "mean":
        return attention[layer_idx][0].mean(dim=0)
    elif type == "max":
        return attention[layer_idx][0].max(dim=0)[0]
    elif type == "heads":
        return attention[layer_idx][0]
    
def cls_attention_at_layer(attention, layer_idx, type: Literal["mean", "max", "heads"] = "mean"):
    if type == "mean":
        return attention[layer_idx][0].mean(dim=0)[0, 1:]
    elif type == "max":
        return attention[layer_idx][0].max(dim=0)[0][0, 1:]
    elif type == "heads":
        return attention[layer_idx][0][:, 0, 1:]
    
print_separator('Full attention at layer 0')
print(full_attention_at_layer(layer_attentions, 0, type="mean").shape)
print(full_attention_at_layer(layer_attentions, 0, type="max").shape)
print(full_attention_at_layer(layer_attentions, 0, type="heads").shape)

print_separator('CLS attention at layer 0')
print(cls_attention_at_layer(layer_attentions, 0, type="mean").shape)
print(cls_attention_at_layer(layer_attentions, 0, type="max").shape)
print(cls_attention_at_layer(layer_attentions, 0, type="heads").shape)

In [ ]:
import base64, io, os, cv2, importlib
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image

PATCH_SIZE = 16
GRID       = 14
IMG_SIZE   = 224

# patch centres in data coords (row-major)
_PX = [c * PATCH_SIZE + PATCH_SIZE // 2 for c in range(GRID)]
_PY = [r * PATCH_SIZE + PATCH_SIZE // 2 for r in range(GRID)]


def _arr_to_uri(arr):
    """numpy uint8 → inline PNG data-URI for layout_image."""
    buf = io.BytesIO()
    Image.fromarray(arr.astype(np.uint8)).save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def _load_img(path):
    return np.array(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))


def _get_attn_grid(attentions, layer, token_idx, agg):
    attn = attentions[layer][0]   # (heads, 197, 197)
    row  = (attn.mean(0) if agg == "mean" else attn.max(0)[0])[token_idx, 1:]
    return row.detach().cpu().numpy().reshape(GRID, GRID)


def visualize_attention_clickable(all_images_list, outputs_dict):
    widgets     = importlib.import_module("ipywidgets")
    ipy_display = importlib.import_module("IPython.display")

    # ── controls ──────────────────────────────────────────────────────────────
    img_opts  = [(os.path.basename(p), p) for p in all_images_list]
    dropdown  = widgets.Dropdown(options=img_opts, description="Image:")
    n_layers  = len(outputs_dict[all_images_list[0]].attentions)
    slider    = widgets.IntSlider(value=0, min=0, max=n_layers - 1,
                                  description="Layer:", continuous_update=False)
    reset_btn = widgets.Button(description="Reset → CLS", button_style="warning")
    info_lbl  = widgets.Label("Token: CLS")
    debug_lbl = widgets.Label("Debug: waiting for first click…")

    state = {"token_idx": 0, "patch_idx": None}

    # ── figure ────────────────────────────────────────────────────────────────
    blank = np.zeros((GRID, GRID), dtype=np.float32)

    fig = go.FigureWidget(make_subplots(
        rows=1, cols=2,
        subplot_titles=["Attention (MEAN)", "Attention (MAX)"],
        horizontal_spacing=0.08,
    ))

    # Heatmaps only — no go.Image, so nothing blocks click events
    fig.add_trace(go.Heatmap(
        z=blank, x=_PX, y=_PY,
        opacity=0.55, colorscale="Jet", showscale=False,
        name="heat_mean",
    ), row=1, col=1)

    fig.add_trace(go.Heatmap(
        z=blank, x=_PX, y=_PY,
        opacity=0.55, colorscale="Jet", showscale=False,
        name="heat_max",
    ), row=1, col=2)

    # y-axis: inverted so row-0 is at the top (image convention)
    for xax in ("xaxis", "xaxis2"):
        fig.layout[xax].update(range=[0, IMG_SIZE], showticklabels=False,
                               constrain="domain")
    for yax in ("yaxis", "yaxis2"):
        fig.layout[yax].update(range=[IMG_SIZE, 0], showticklabels=False,
                               scaleanchor="x")

    # Background photos as layout_image — purely decorative, never eat clicks
    init_uri = _arr_to_uri(_load_img(all_images_list[0]))
    for xref, yref in [("x", "y"), ("x2", "y2")]:
        fig.add_layout_image(dict(
            source=init_uri,
            xref=xref, yref=yref,
            x=0, y=0,
            sizex=IMG_SIZE, sizey=IMG_SIZE,
            sizing="stretch",
            layer="below",
            opacity=1.0,
        ))

    fig.update_layout(height=430, margin=dict(l=20, r=20, t=55, b=20))

    # ── redraw ────────────────────────────────────────────────────────────────
    def _redraw():
        path  = dropdown.value
        tok   = state["token_idx"]
        layer = slider.value
        attn  = outputs_dict[path].attentions

        mean_g = _get_attn_grid(attn, layer, tok, "mean")
        max_g  = _get_attn_grid(attn, layer, tok, "max")
        uri    = _arr_to_uri(_load_img(path))

        with fig.batch_update():
            fig.data[0].z = mean_g
            fig.data[1].z = max_g
            fig.layout.images[0].source = uri
            fig.layout.images[1].source = uri

        pi = state["patch_idx"]
        label = "CLS" if tok == 0 else f"patch {pi} (row {pi//GRID}, col {pi%GRID})"
        info_lbl.value = f"Token: {label}  — click a patch | Reset → CLS"

    # ── click on heatmap (works now that go.Image is gone) ────────────────────
    def _on_click(trace, points, selector):
        print(f"[CLICK] trace={trace.name!r}  "
              f"point_inds={points.point_inds}  xs={points.xs}  ys={points.ys}")

        if not points.xs:
            debug_lbl.value = "⚠ xs empty — try clicking directly on the coloured heatmap area"
            return

        px  = float(points.xs[0])
        py  = float(points.ys[0])
        col = min(int(px) // PATCH_SIZE, GRID - 1)
        row = min(int(py) // PATCH_SIZE, GRID - 1)
        patch_idx = row * GRID + col
        token_idx = patch_idx + 1

        state["patch_idx"] = patch_idx
        state["token_idx"] = token_idx
        debug_lbl.value = (f"✓ px=({px:.0f}, {py:.0f})  →  "
                           f"patch {patch_idx}  (row {row}, col {col})  →  token {token_idx}")
        _redraw()

    fig.data[0].on_click(_on_click)
    fig.data[1].on_click(_on_click)

    # ── other controls ────────────────────────────────────────────────────────
    def _reset(_):
        state["patch_idx"] = None
        state["token_idx"] = 0
        debug_lbl.value = "Reset → CLS"
        _redraw()

    reset_btn.on_click(_reset)
    dropdown.observe(lambda _: _redraw(), names="value")
    slider.observe(lambda _:   _redraw(), names="value")

    ipy_display.display(widgets.VBox([
        widgets.HBox([dropdown, slider, reset_btn]),
        info_lbl, debug_lbl, fig,
    ]))
    _redraw()

In [ ]:
# Example: single image with interactive controls
i = 2
# visualize_attention(outputs[all_images[i]], all_images[i])

# Example: interactive image selector with per-layer visualization
visualize_attention_clickable(all_images, outputs)

# Attention Rollout

* **Attention rollout:**
    * Feed image into ViT
    * Attention layers $[A^l_h]$: layer $l$ has $h$ attention heads with size $(N+1)\times(N+1)$ ($N$ is the number of image patch tokens + 1 CLS token)
    * Aggregate (can be done with mean, max, min...) the attention over heads so that $[A^l_h]$ is reduced to $[A^l]$
    * Compute the attention rollout (estimating the flow of attention in the ViT network):
$$A_{rollout}^l = (A^l + I)A_{rollout}^{l-1}, \text{ for } i > 1$$
$$A_{rollout}^1 = A^1 + I$$

In [ ]:
from typing import Literal

def compute_attention_rollout(attentions, mode: Literal["mean", "max", "heads"] = "mean"):
    A = np.eye(197)  
    for layer in range(len(attentions)):
        attn = full_attention_at_layer(attentions, layer, type=mode).detach().numpy()
        attn = (attn + np.eye(197)) / attn.sum(axis=-1, keepdims=True)
        A = attn @ A

    rollout_cls = A[0, 1:]
    return rollout_cls

def _plot_rollout_single(output, image_path):
    img_arr       = _load_img(image_path)
    predicted_label = os.path.basename(image_path).split("_")[0]
    layer_attentions = output.attentions

    rollout_mean = compute_attention_rollout(layer_attentions, mode="mean").reshape(GRID, GRID)
    rollout_max  = compute_attention_rollout(layer_attentions, mode="max").reshape(GRID, GRID)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Attention Rollout (MEAN)", "Attention Rollout (MAX)"],
        horizontal_spacing=0.08,
    )

    fig.add_trace(go.Heatmap(
        z=rollout_mean, x=_PX, y=_PY,
        opacity=0.55, colorscale="Jet", showscale=False,
        zmin=0, zmax=float(rollout_mean.max()),
    ), row=1, col=1)

    fig.add_trace(go.Heatmap(
        z=rollout_max, x=_PX, y=_PY,
        opacity=0.55, colorscale="Jet", showscale=False,
        zmin=0, zmax=float(rollout_max.max()),
    ), row=1, col=2)

    img_uri = _arr_to_uri(img_arr)
    for xref, yref in [("x", "y"), ("x2", "y2")]:
        fig.add_layout_image(dict(
            source=img_uri,
            xref=xref, yref=yref,
            x=0, y=0,
            sizex=IMG_SIZE, sizey=IMG_SIZE,
            sizing="stretch",
            layer="below",
            opacity=1.0,
        ))

    for xax in ("xaxis", "xaxis2"):
        fig.layout[xax].update(range=[0, IMG_SIZE], showticklabels=False,
                               constrain="domain")
    for yax in ("yaxis", "yaxis2"):
        fig.layout[yax].update(range=[IMG_SIZE, 0], showticklabels=False,
                               scaleanchor="x")

    fig.update_layout(
        title=f"Attention Rollout: Mean vs Max   |   Predicted: {predicted_label}",
        margin=dict(l=30, r=30, t=60, b=30),
    )
    return fig


def plot_rollout_attention(all_images_list, outputs_dict):
    widgets     = importlib.import_module("ipywidgets")
    ipy_display = importlib.import_module("IPython.display")

    img_opts = [(os.path.basename(p), p) for p in all_images_list]
    dropdown = widgets.Dropdown(options=img_opts, description="Image:")
    out      = widgets.Output()

    def _render(*_):
        with out:
            ipy_display.clear_output(wait=True)
            _plot_rollout_single(outputs_dict[dropdown.value], dropdown.value).show()

    dropdown.observe(_render, names="value")
    ipy_display.display(widgets.VBox([widgets.HBox([dropdown]), out]))
    _render()

# Example: interactive image selection with rollout visualization
plot_rollout_attention(all_images, outputs)